# 05 Final Load Preparation

This notebook prepares the final Tableau-ready data pack for the capstone submission. It loads the cleaned market dataset plus the saved KPI, EDA, and statistical-analysis outputs, then exports business-friendly CSVs for dashboarding.

Generated files:

- `data/processed/tableau_stock_level.csv`
- `data/processed/tableau_sector_level.csv`
- `data/processed/tableau_yearly_trends.csv`
- `data/processed/tableau_risk_segments.csv`
- `data/processed/tableau_recommendation_view.csv`
- `outputs/tables/final_load_validation.csv`

The notebook keeps the final layer transparent by using the repository's existing outputs as inputs instead of recomputing the whole project from scratch. Numbers shown here should therefore match the versioned tables in `outputs/tables/`.

In [1]:
from __future__ import annotations

from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display


def resolve_repo_root() -> Path:
    current = Path.cwd().resolve()
    candidates = [current, current.parent]
    for candidate in candidates:
        if (candidate / 'scripts').exists() and (candidate / 'data').exists():
            return candidate
    raise FileNotFoundError('Could not resolve the repository root from the current working directory.')


REPO_ROOT = resolve_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from scripts.eda_analysis import build_market_daily, build_sector_daily

DATA_DIR = REPO_ROOT / 'data' / 'processed'
OUTPUTS_DIR = REPO_ROOT / 'outputs' / 'tables'
NOTEBOOK_NAME = '05_final_load_prep.ipynb'

pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', '{:,.4f}'.format)

In [2]:
clean_df = pd.read_csv(DATA_DIR / 'nifty50_cleaned.csv', parse_dates=['date'])
stock_kpis = pd.read_csv(OUTPUTS_DIR / 'stock_kpis.csv')
sector_kpis = pd.read_csv(OUTPUTS_DIR / 'sector_kpis.csv')
yearly_market_summary = pd.read_csv(OUTPUTS_DIR / 'yearly_market_summary.csv')
covid_period_summary = pd.read_csv(OUTPUTS_DIR / 'covid_period_summary.csv')
risk_summary = pd.read_csv(OUTPUTS_DIR / 'risk_summary.csv')
stock_segments = pd.read_csv(OUTPUTS_DIR / 'stock_segments.csv')
recommendation_evidence = pd.read_csv(OUTPUTS_DIR / 'recommendation_evidence.csv')
statistical_test_results = pd.read_csv(OUTPUTS_DIR / 'statistical_test_results.csv')

for column in ['missing_price_flag', 'missing_volume_flag', 'invalid_ohlc_flag', 'outlier_return_flag']:
    if column in clean_df.columns and clean_df[column].dtype == object:
        clean_df[column] = clean_df[column].astype(str).str.lower().eq('true')

clean_df['analysis_return'] = clean_df['daily_return'].where(~clean_df['outlier_return_flag'].fillna(False))
clean_df = clean_df.sort_values(['source_symbol', 'date']).reset_index(drop=True)

input_summary = pd.DataFrame(
    [
        {'artifact': 'cleaned_dataset', 'rows': clean_df.shape[0], 'columns': clean_df.shape[1]},
        {'artifact': 'stock_kpis', 'rows': stock_kpis.shape[0], 'columns': stock_kpis.shape[1]},
        {'artifact': 'sector_kpis', 'rows': sector_kpis.shape[0], 'columns': sector_kpis.shape[1]},
        {'artifact': 'yearly_market_summary', 'rows': yearly_market_summary.shape[0], 'columns': yearly_market_summary.shape[1]},
        {'artifact': 'covid_period_summary', 'rows': covid_period_summary.shape[0], 'columns': covid_period_summary.shape[1]},
        {'artifact': 'risk_summary', 'rows': risk_summary.shape[0], 'columns': risk_summary.shape[1]},
        {'artifact': 'stock_segments', 'rows': stock_segments.shape[0], 'columns': stock_segments.shape[1]},
        {'artifact': 'recommendation_evidence', 'rows': recommendation_evidence.shape[0], 'columns': recommendation_evidence.shape[1]},
        {'artifact': 'statistical_test_results', 'rows': statistical_test_results.shape[0], 'columns': statistical_test_results.shape[1]},
    ]
)

display(input_summary)
print('Date range:', clean_df['date'].min().date(), 'to', clean_df['date'].max().date())
print('Stable stock count:', clean_df['source_symbol'].nunique())
print('Historical row-level symbol count:', clean_df['symbol'].nunique())
print('Sector count:', clean_df['industry'].nunique(dropna=True))

,artifact,rows,columns
0,cleaned_dataset,235192,36
1,stock_kpis,49,20
2,sector_kpis,13,18
3,yearly_market_summary,22,14
4,covid_period_summary,3,14
5,risk_summary,49,13
6,stock_segments,49,21
7,recommendation_evidence,5,7
8,statistical_test_results,7,23


Date range: 2000-01-03 to 2021-04-30
Stable stock count: 49
Historical row-level symbol count: 65
Sector count: 13


In [3]:
TRADING_DAYS_PER_YEAR = 252
COVID_WINDOWS = [
    ('Pre-COVID Baseline', '2019-01-01', '2020-02-14'),
    ('Crash Phase', '2020-02-17', '2020-03-31'),
    ('Recovery Phase', '2020-04-01', '2020-12-31'),
]

ACTION_MAP = {
    'Stable Compounders': 'Buy',
    'Defensive / Low Volatility': 'Watch',
    'Liquid Trading Candidates': 'Trade Candidate',
    'High Growth High Risk': 'Trade Candidate',
    'Weak Risk-Return Candidates': 'Avoid',
}

THEME_MAP = {
    'Stable Compounders': 'Core Compounder Basket',
    'Defensive / Low Volatility': 'Defensive Regime Watchlist',
    'Liquid Trading Candidates': 'Execution-First Trading Watchlist',
    'High Growth High Risk': 'Risk-Capped Growth Sleeve',
    'Weak Risk-Return Candidates': 'Underweight / Avoid List',
}

ACTION_RATIONALE_MAP = {
    'Stable Compounders': 'High-conviction core candidates based on return quality, risk discipline, and supporting opportunity scores.',
    'Defensive / Low Volatility': 'Monitor as resilient lower-volatility names, especially when broad market momentum softens.',
    'Liquid Trading Candidates': 'Useful for trading-oriented monitoring because liquidity is strong, but execution should still be paired with risk filters.',
    'High Growth High Risk': 'Tactical upside candidates that require tighter risk control because volatility and downside measures are elevated.',
    'Weak Risk-Return Candidates': 'Deprioritize because return quality, drawdown behavior, or risk-adjusted performance is not attractive enough for the final story.',
}


def compute_return_metrics(returns: pd.Series) -> dict[str, float]:
    returns = returns.dropna().astype(float)
    if returns.empty:
        return {
            'Cumulative Return (%)': np.nan,
            'Annualized Return (%)': np.nan,
            'Volatility (%)': np.nan,
            'Average Daily Return (%)': np.nan,
            'Max Drawdown (%)': np.nan,
            'Risk-Adjusted Return': np.nan,
            'Trading Days': 0,
        }

    compounded_growth = (1 + returns).prod()
    cumulative_return = compounded_growth - 1
    trading_days = int(returns.shape[0])
    annualized_return = np.nan
    if compounded_growth > 0 and trading_days > 0:
        annualized_return = compounded_growth ** (TRADING_DAYS_PER_YEAR / trading_days) - 1

    volatility = returns.std(ddof=1) * np.sqrt(TRADING_DAYS_PER_YEAR)
    average_daily_return = returns.mean() * 100
    wealth_index = (1 + returns).cumprod()
    running_peak = wealth_index.cummax()
    max_drawdown = (wealth_index / running_peak - 1).min()
    risk_adjusted_return = np.nan
    if pd.notna(volatility) and volatility > 0 and pd.notna(annualized_return):
        risk_adjusted_return = annualized_return / volatility

    return {
        'Cumulative Return (%)': cumulative_return * 100,
        'Annualized Return (%)': annualized_return * 100 if pd.notna(annualized_return) else np.nan,
        'Volatility (%)': volatility * 100 if pd.notna(volatility) else np.nan,
        'Average Daily Return (%)': average_daily_return,
        'Max Drawdown (%)': max_drawdown * 100,
        'Risk-Adjusted Return': risk_adjusted_return,
        'Trading Days': trading_days,
    }


def add_index_columns(frame: pd.DataFrame, group_column: str | None = None) -> pd.DataFrame:
    ordered = frame.sort_values(['date'] if group_column is None else [group_column, 'date']).copy()

    if group_column is None:
        ordered['equal_weighted_index'] = (1 + ordered['daily_return'].fillna(0)).cumprod() * 100
        ordered['ma_20'] = ordered['equal_weighted_index'].rolling(20).mean()
        ordered['ma_60'] = ordered['equal_weighted_index'].rolling(60).mean()
    else:
        ordered['equal_weighted_index'] = ordered.groupby(group_column)['daily_return'].transform(
            lambda values: (1 + values.fillna(0)).cumprod() * 100
        )
        ordered['ma_20'] = ordered.groupby(group_column)['equal_weighted_index'].transform(lambda values: values.rolling(20).mean())
        ordered['ma_60'] = ordered.groupby(group_column)['equal_weighted_index'].transform(lambda values: values.rolling(60).mean())

    ordered['trend_signal'] = np.where(ordered['ma_20'] >= ordered['ma_60'], 'Uptrend', 'Downtrend')
    ordered.loc[ordered['ma_20'].isna() | ordered['ma_60'].isna(), 'trend_signal'] = 'Insufficient History'
    return ordered


def build_sector_covid_summary(sector_daily: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for industry, group in sector_daily.groupby('industry', sort=True):
        row = {'industry': industry}
        for period_name, start_date, end_date in COVID_WINDOWS:
            window = group.loc[(group['date'] >= start_date) & (group['date'] <= end_date)].copy()
            metrics = compute_return_metrics(window['daily_return'])
            prefix = period_name.replace('-', ' ').replace(' ', '_').lower()
            row[f'{prefix}_return_percent'] = metrics['Cumulative Return (%)']
            row[f'{prefix}_volatility_percent'] = metrics['Volatility (%)']
            row[f'{prefix}_turnover_cr'] = window['turnover_cr'].mean()
            row[f'{prefix}_delivery_percent'] = window['deliverable_percent'].mean() * 100
        row['recovery_minus_crash_return_pp'] = row['recovery_phase_return_percent'] - row['crash_phase_return_percent']
        rows.append(row)
    return pd.DataFrame(rows)


def validate_tableau_dataset(name: str, df: pd.DataFrame, key_columns: list[str], critical_columns: list[str]) -> dict[str, object]:
    duplicate_keys = int(df.duplicated(subset=key_columns).sum()) if key_columns else 0
    missing_cells = int(df.isna().sum().sum())
    missing_columns = ', '.join(df.columns[df.isna().any()].tolist()) or 'No missing columns'
    index_like_columns = [column for column in df.columns if column.lower().startswith('unnamed')]

    date_min = str(clean_df['date'].min().date())
    date_max = str(clean_df['date'].max().date())
    if 'Date' in df.columns:
        date_min = str(pd.to_datetime(df['Date']).min().date())
        date_max = str(pd.to_datetime(df['Date']).max().date())
    elif {'Start Date', 'End Date'}.issubset(df.columns):
        date_min = str(pd.to_datetime(df['Start Date']).min().date())
        date_max = str(pd.to_datetime(df['End Date']).max().date())

    stock_count = int(df['Stock Symbol'].nunique()) if 'Stock Symbol' in df.columns else 0
    sector_count = int(df['Sector / Industry'].replace('All Sectors', pd.NA).dropna().nunique()) if 'Sector / Industry' in df.columns else 0

    kpi_availability = round(float(df[critical_columns].notna().mean().mean() * 100), 2) if critical_columns else 100.0

    return {
        'dataset_name': name,
        'rows': int(df.shape[0]),
        'columns': int(df.shape[1]),
        'duplicate_key_rows': duplicate_keys,
        'missing_value_cells': missing_cells,
        'columns_with_missing_values': missing_columns,
        'date_range_start': date_min,
        'date_range_end': date_max,
        'stock_count': stock_count,
        'sector_count': sector_count,
        'kpi_availability_percent': kpi_availability,
        'index_like_columns': ', '.join(index_like_columns) or 'No index-like columns',
    }

In [4]:
market_daily = add_index_columns(build_market_daily(clean_df))
sector_daily = add_index_columns(build_sector_daily(clean_df), group_column='industry')

market_trends = market_daily.assign(
    view_level='Market',
    industry='All Sectors',
    quarter=market_daily['date'].dt.quarter,
    month=market_daily['date'].dt.month,
).rename(
    columns={
        'date': 'Date',
        'year': 'Year',
        'industry': 'Sector / Industry',
        'daily_return': 'Daily Return',
        'equal_weighted_index': 'Equal Weighted Index',
        'ma_20': '20D Moving Average',
        'ma_60': '60D Moving Average',
        'trend_signal': 'Trend Signal',
        'volume': 'Volume',
        'turnover_cr': 'Turnover (Cr)',
        'deliverable_percent': 'Average Delivery (%)',
        'vwap_gap': 'Average VWAP Gap',
        'stock_count': 'Stock Count',
        'view_level': 'View Level',
        'quarter': 'Quarter',
        'month': 'Month',
    }
)
market_trends['Average Delivery (%)'] = market_trends['Average Delivery (%)'] * 100

sector_trends = sector_daily.assign(
    view_level='Sector',
    quarter=sector_daily['date'].dt.quarter,
    month=sector_daily['date'].dt.month,
).rename(
    columns={
        'date': 'Date',
        'year': 'Year',
        'industry': 'Sector / Industry',
        'daily_return': 'Daily Return',
        'equal_weighted_index': 'Equal Weighted Index',
        'ma_20': '20D Moving Average',
        'ma_60': '60D Moving Average',
        'trend_signal': 'Trend Signal',
        'volume': 'Volume',
        'turnover_cr': 'Turnover (Cr)',
        'deliverable_percent': 'Average Delivery (%)',
        'stock_count': 'Stock Count',
        'view_level': 'View Level',
        'quarter': 'Quarter',
        'month': 'Month',
    }
)
sector_trends['Average Delivery (%)'] = sector_trends['Average Delivery (%)'] * 100
sector_vwap_gap = (
    clean_df.groupby(['industry', 'date'], as_index=False)['vwap_gap']
    .mean()
    .rename(columns={'industry': 'Sector / Industry', 'date': 'Date', 'vwap_gap': 'Average VWAP Gap'})
)
sector_trends = sector_trends.merge(sector_vwap_gap, on=['Sector / Industry', 'Date'], how='left')

tableau_yearly_trends = pd.concat([market_trends, sector_trends], ignore_index=True, sort=False)
tableau_yearly_trends = tableau_yearly_trends[
    [
        'View Level', 'Sector / Industry', 'Date', 'Year', 'Quarter', 'Month', 'Daily Return',
        'Equal Weighted Index', '20D Moving Average', '60D Moving Average', 'Trend Signal',
        'Volume', 'Turnover (Cr)', 'Average Delivery (%)', 'Average VWAP Gap', 'Stock Count'
    ]
].sort_values(['View Level', 'Sector / Industry', 'Date']).reset_index(drop=True)

sector_covid_summary = build_sector_covid_summary(sector_daily)
sector_best_stock = (
    stock_kpis.sort_values(['industry', 'opportunity_score', 'annualized_return_percent'], ascending=[True, False, False])
    .groupby('industry', as_index=False)
    .first()[['industry', 'symbol', 'company_name']]
    .rename(columns={'symbol': 'Best Stock by Opportunity', 'company_name': 'Best Stock Company'})
)

sector_level = (
    sector_kpis
    .merge(sector_covid_summary, on='industry', how='left')
    .merge(sector_best_stock, on='industry', how='left')
    .rename(
        columns={
            'industry': 'Sector / Industry',
            'constituent_stock_count': 'Constituent Stock Count',
            'trading_days': 'Trading Days',
            'cumulative_return_percent': 'Cumulative Return (%)',
            'annualized_return_percent': 'Annualized Return (%)',
            'volatility_percent': 'Volatility (%)',
            'average_daily_return': 'Average Daily Return (%)',
            'max_drawdown_percent': 'Max Drawdown (%)',
            'risk_adjusted_return': 'Risk-Adjusted Return',
            'average_volume': 'Average Volume',
            'average_turnover_cr': 'Average Turnover (Cr)',
            'average_delivery_percent': 'Average Delivery (%)',
            'positive_return_day_ratio': 'Positive Return Days (%)',
            'close_above_vwap_ratio': 'Close Above VWAP (%)',
            'average_annual_return_percent': 'Average Annual Return (%)',
            'liquidity_score': 'Liquidity Score',
            'investor_confidence_score': 'Investor Confidence Score',
            'opportunity_score': 'Opportunity Score',
            'pre_covid_baseline_return_percent': 'Pre-COVID Return (%)',
            'crash_phase_return_percent': 'COVID Crash Return (%)',
            'recovery_phase_return_percent': 'COVID Recovery Return (%)',
            'pre_covid_baseline_volatility_percent': 'Pre-COVID Volatility (%)',
            'crash_phase_volatility_percent': 'COVID Crash Volatility (%)',
            'recovery_phase_volatility_percent': 'COVID Recovery Volatility (%)',
            'recovery_minus_crash_return_pp': 'Recovery Minus Crash (pp)',
        }
    )
)
tableau_sector_level = sector_level[
    [
        'Sector / Industry', 'Constituent Stock Count', 'Trading Days', 'Cumulative Return (%)', 'Annualized Return (%)',
        'Average Annual Return (%)', 'Volatility (%)', 'Average Daily Return (%)', 'Max Drawdown (%)',
        'Risk-Adjusted Return', 'Average Volume', 'Average Turnover (Cr)', 'Average Delivery (%)',
        'Positive Return Days (%)', 'Close Above VWAP (%)', 'Liquidity Score', 'Investor Confidence Score',
        'Opportunity Score', 'Best Stock by Opportunity', 'Best Stock Company', 'Pre-COVID Return (%)',
        'COVID Crash Return (%)', 'COVID Recovery Return (%)', 'Pre-COVID Volatility (%)',
        'COVID Crash Volatility (%)', 'COVID Recovery Volatility (%)', 'Recovery Minus Crash (pp)'
    ]
].sort_values(['Opportunity Score', 'Annualized Return (%)'], ascending=[False, False]).reset_index(drop=True)

stock_level = (
    stock_kpis
    .merge(risk_summary[['symbol', 'risk_score', 'risk_bucket']], on='symbol', how='left')
    .merge(stock_segments[['symbol', 'segment', 'segment_score', 'segment_rationale']], on='symbol', how='left')
    .assign(
        recommendation_action=lambda frame: frame['segment'].map(ACTION_MAP),
        recommendation_theme=lambda frame: frame['segment'].map(THEME_MAP),
        recommendation_rationale=lambda frame: frame['segment'].map(ACTION_RATIONALE_MAP),
    )
    .rename(
        columns={
            'symbol': 'Stock Symbol',
            'company_name': 'Company Name',
            'industry': 'Sector / Industry',
            'start_date': 'Start Date',
            'end_date': 'End Date',
            'trading_days': 'Trading Days',
            'cumulative_return_percent': 'Cumulative Return (%)',
            'annualized_return_percent': 'Annualized Return (%)',
            'volatility_percent': 'Volatility (%)',
            'average_daily_return': 'Average Daily Return (%)',
            'max_drawdown_percent': 'Max Drawdown (%)',
            'risk_adjusted_return': 'Risk-Adjusted Return',
            'average_volume': 'Average Volume',
            'average_turnover_cr': 'Average Turnover (Cr)',
            'average_delivery_percent': 'Average Delivery (%)',
            'positive_return_day_ratio': 'Positive Return Days (%)',
            'close_above_vwap_ratio': 'Close Above VWAP (%)',
            'liquidity_score': 'Liquidity Score',
            'investor_confidence_score': 'Investor Confidence Score',
            'opportunity_score': 'Opportunity Score',
            'risk_score': 'Risk Score',
            'risk_bucket': 'Risk Bucket',
            'segment': 'Segment',
            'segment_score': 'Segment Score',
            'segment_rationale': 'Segment Rationale',
            'recommendation_action': 'Recommendation Action',
            'recommendation_theme': 'Recommendation Theme',
            'recommendation_rationale': 'Recommendation Rationale',
        }
    )
)
tableau_stock_level = stock_level[
    [
        'Stock Symbol', 'Company Name', 'Sector / Industry', 'Start Date', 'End Date', 'Trading Days',
        'Cumulative Return (%)', 'Annualized Return (%)', 'Volatility (%)', 'Average Daily Return (%)',
        'Max Drawdown (%)', 'Risk-Adjusted Return', 'Average Volume', 'Average Turnover (Cr)',
        'Average Delivery (%)', 'Positive Return Days (%)', 'Close Above VWAP (%)', 'Liquidity Score',
        'Investor Confidence Score', 'Opportunity Score', 'Risk Score', 'Risk Bucket', 'Segment',
        'Segment Score', 'Recommendation Action', 'Recommendation Theme', 'Recommendation Rationale'
    ]
].sort_values(['Opportunity Score', 'Annualized Return (%)'], ascending=[False, False]).reset_index(drop=True)

tableau_risk_segments = (
    stock_segments
    .merge(risk_summary[['symbol', 'risk_score']], on='symbol', how='left')
    .merge(stock_kpis[['symbol', 'investor_confidence_score', 'opportunity_score']], on='symbol', how='left')
    .assign(
        recommendation_action=lambda frame: frame['segment'].map(ACTION_MAP),
        recommendation_theme=lambda frame: frame['segment'].map(THEME_MAP),
    )
    .rename(
        columns={
            'symbol': 'Stock Symbol',
            'company_name': 'Company Name',
            'industry': 'Sector / Industry',
            'segment': 'Segment',
            'segment_score': 'Segment Score',
            'annualized_return_percent': 'Annualized Return (%)',
            'volatility_percent': 'Volatility (%)',
            'risk_adjusted_return': 'Risk-Adjusted Return',
            'liquidity_score': 'Liquidity Score',
            'average_delivery_percent': 'Average Delivery (%)',
            'max_drawdown_percent': 'Max Drawdown (%)',
            'downside_volatility_percent': 'Downside Volatility (%)',
            'historical_var_95_percent': 'Historical VaR 95 (%)',
            'historical_cvar_95_percent': 'Historical CVaR 95 (%)',
            'risk_bucket': 'Risk Bucket',
            'risk_score': 'Risk Score',
            'stable_score': 'Stable Compounder Score',
            'growth_risk_score': 'Growth-Risk Score',
            'liquid_trading_score': 'Liquid Trading Score',
            'weak_score': 'Weak Risk-Return Score',
            'defensive_score': 'Defensive Score',
            'segment_rationale': 'Segment Rationale',
            'investor_confidence_score': 'Investor Confidence Score',
            'opportunity_score': 'Opportunity Score',
            'recommendation_action': 'Recommendation Action',
            'recommendation_theme': 'Recommendation Theme',
        }
    )
)
tableau_risk_segments = tableau_risk_segments[
    [
        'Stock Symbol', 'Company Name', 'Sector / Industry', 'Segment', 'Segment Score', 'Recommendation Action',
        'Recommendation Theme', 'Risk Bucket', 'Risk Score', 'Annualized Return (%)', 'Volatility (%)',
        'Risk-Adjusted Return', 'Liquidity Score', 'Investor Confidence Score', 'Opportunity Score',
        'Average Delivery (%)', 'Max Drawdown (%)', 'Downside Volatility (%)', 'Historical VaR 95 (%)',
        'Historical CVaR 95 (%)', 'Stable Compounder Score', 'Growth-Risk Score', 'Liquid Trading Score',
        'Weak Risk-Return Score', 'Defensive Score', 'Segment Rationale'
    ]
].sort_values(['Risk Score', 'Opportunity Score'], ascending=[False, False]).reset_index(drop=True)

recommendation_view = tableau_stock_level.copy()
recommendation_view['Action Sort Order'] = recommendation_view['Recommendation Action'].map({
    'Buy': 1,
    'Watch': 2,
    'Trade Candidate': 3,
    'Avoid': 4,
})
recommendation_view['Opportunity Rank'] = recommendation_view['Opportunity Score'].rank(method='dense', ascending=False).astype(int)
recommendation_view['Recommendation Label'] = recommendation_view['Recommendation Action'] + ' - ' + recommendation_view['Recommendation Theme']
recommendation_view['Why This Action'] = recommendation_view['Recommendation Rationale']
tableau_recommendation_view = recommendation_view[
    [
        'Action Sort Order', 'Opportunity Rank', 'Recommendation Label', 'Recommendation Action', 'Recommendation Theme',
        'Stock Symbol', 'Company Name', 'Sector / Industry', 'Segment', 'Risk Bucket', 'Opportunity Score',
        'Liquidity Score', 'Investor Confidence Score', 'Annualized Return (%)', 'Volatility (%)',
        'Risk-Adjusted Return', 'Max Drawdown (%)', 'Average Delivery (%)', 'Average Turnover (Cr)',
        'Why This Action'
    ]
].sort_values(['Action Sort Order', 'Opportunity Rank']).reset_index(drop=True)

display(tableau_stock_level.head(10))
display(tableau_sector_level.head(10))
display(tableau_yearly_trends.head(10))

,Stock Symbol,Company Name,Sector / Industry,Start Date,End Date,Trading Days,Cumulative Return (%),Annualized Return (%),Volatility (%),Average Daily Return (%),Max Drawdown (%),Risk-Adjusted Return,Average Volume,Average Turnover (Cr),Average Delivery (%),Positive Return Days (%),Close Above VWAP (%),Liquidity Score,Investor Confidence Score,Opportunity Score,Risk Score,Risk Bucket,Segment,Segment Score,Recommendation Action,Recommendation Theme,Recommendation Rationale
0,TCS,Tata Consultancy Services Ltd.,IT,2004-08-25,2021-04-30,3939,"1,907.2149",21.1530,23.2774,0.0869,-51.8935,0.9087,"1,676,761.9505",295.2102,54.9429,48.8765,47.9826,66.3265,77.5510,91.7347,16.3265,Low,Stable Compounders,89.5510,Buy,Core Compounder Basket,High-conviction core candidates based on retur...
1,RELIANCE,Reliance Industries Ltd.,ENERGY,2000-01-03,2021-04-30,5086,"2,783.7777",18.1239,26.1732,0.0797,-45.3477,0.6925,"5,583,027.5424",607.6715,43.5039,49.4534,44.6099,89.7959,36.2245,84.7959,29.0816,Low,Liquid Trading Candidates,92.1429,Trade Candidate,Execution-First Trading Watchlist,Useful for trading-oriented monitoring because...
2,ASIANPAINT,Asian Paints Ltd.,CONSUMER GOODS,2000-01-03,2021-04-30,5018,"1,400.9987",14.5716,20.3237,0.0622,-42.0882,0.7170,"509,672.0869",69.4292,62.5572,47.6442,47.8703,18.3673,85.0000,80.6122,6.6327,Low,Defensive / Low Volatility,93.0612,Watch,Defensive Regime Watchlist,"Monitor as resilient lower-volatility names, e..."
3,HDFC,Housing Development Finance Corporation Ltd.,FINANCIAL SERVICES,2000-01-03,2021-04-30,5035,891.6615,12.1677,26.7259,0.0597,-52.9465,0.4553,"1,848,186.9159",262.0759,65.3088,47.4746,52.2239,63.2653,90.8163,79.6939,37.2449,Medium,Stable Compounders,78.4490,Buy,Core Compounder Basket,High-conviction core candidates based on retur...
4,BAJAJ-AUTO,Bajaj Auto Ltd.,AUTOMOBILE,2008-05-26,2021-04-30,3046,517.0654,16.2479,22.8687,0.0701,-26.7537,0.7105,"411,463.9169",96.4402,50.9401,47.8451,47.0331,24.4898,60.9184,79.6429,10.2041,Low,Stable Compounders,86.6939,Buy,Core Compounder Basket,High-conviction core candidates based on retur...
5,HDFCBANK,HDFC Bank Ltd.,FINANCIAL SERVICES,2000-01-03,2021-04-30,4971,618.3112,10.5121,21.9575,0.0492,-44.9848,0.4787,"2,102,579.5156",244.8837,59.5003,47.2296,51.3004,66.3265,81.2755,79.0306,10.2041,Low,Defensive / Low Volatility,88.5714,Watch,Defensive Regime Watchlist,"Monitor as resilient lower-volatility names, e..."
6,MARUTI,Maruti Suzuki India Ltd.,AUTOMOBILE,2003-07-09,2021-04-30,4199,"1,739.2836",19.0959,26.2335,0.0830,-46.8220,0.7279,"1,194,660.7969",239.5307,38.7391,47.7976,44.5448,53.0612,28.3673,77.7551,29.0816,Low,Stable Compounders,77.3878,Buy,Core Compounder Basket,High-conviction core candidates based on retur...
7,HCLTECH,HCL Technologies Ltd.,IT,2000-01-11,2021-04-30,4933,"1,738.6959",16.0371,31.2892,0.0784,-61.2450,0.5125,"1,645,120.7196",107.5213,50.4206,48.1698,47.6415,42.8571,61.0204,74.7959,65.3061,Medium,High Growth High Risk,81.2245,Trade Candidate,Risk-Capped Growth Sleeve,Tactical upside candidates that require tighte...
8,INFY,Infosys Ltd.,IT,2000-01-03,2021-04-30,5026,830.1315,11.8310,26.6345,0.0584,-70.3481,0.4442,"2,622,812.9714",385.5758,53.6366,48.5300,49.5477,78.5714,76.1224,74.5918,45.9184,Medium,Liquid Trading Candidates,82.6531,Trade Candidate,Execution-First Trading Watchlist,Useful for trading-oriented monitoring because...
9,ITC,ITC Ltd.,CONSUMER GOODS,2000-01-03,2021-04-30,5039,472.7203,9.1201,23.0999,0.0452,-53.4014,0.3948,"7,173,165.0207",179.9399,59.1634,47.8892,47.7761,82.6531,79.8980,74.3878,19.3878,Low,Liquid Trading Candidates,85.4082,Trade Candidate,Execution-First Trading Watchlist,Useful for trading-oriented monitoring because...


,Sector / Industry,Constituent Stock Count,Trading Days,Cumulative Return (%),Annualized Return (%),Average Annual Return (%),Volatility (%),Average Daily Return (%),Max Drawdown (%),Risk-Adjusted Return,Average Volume,Average Turnover (Cr),Average Delivery (%),Positive Return Days (%),Close Above VWAP (%),Liquidity Score,Investor Confidence Score,Opportunity Score,Best Stock by Opportunity,Best Stock Company,Pre-COVID Return (%),COVID Crash Return (%),COVID Recovery Return (%),Pre-COVID Volatility (%),COVID Crash Volatility (%),COVID Recovery Volatility (%),Recovery Minus Crash (pp)
0,CEMENT & CEMENT PRODUCTS,3,5200,"2,316.4232",16.6892,20.9005,22.5794,0.0714,-63.3467,0.7391,"697,838.9291",125.6152,59.6477,50.3392,46.5511,19.2308,90.0000,82.6923,GRASIM,Grasim Industries Ltd.,4.6331,-15.3831,45.2695,21.6530,33.3243,24.3311,60.6526
1,PHARMA,3,5271,416.7932,8.1690,9.7250,18.6371,0.0381,-40.7402,0.4383,"4,218,303.7450",315.0887,52.9779,50.6596,44.3837,46.1538,69.6154,78.2692,DRREDDY,Dr. Reddy's Laboratories Ltd.,4.1701,-15.0560,19.9326,16.1748,27.4401,22.1813,34.9886
2,IT,5,5225,791.3042,11.1270,14.8539,23.4519,0.0528,-65.6233,0.4745,"8,902,190.9060",908.9064,48.0960,51.2062,46.5322,73.0769,58.0769,77.3077,TCS,Tata Consultancy Services Ltd.,13.5530,-7.6374,58.6356,14.3830,30.0043,21.5757,66.2729
3,AUTOMOBILE,6,5295,445.5824,8.4098,13.0588,20.6230,0.0405,-60.8575,0.4078,"13,830,720.6796",802.0416,47.4670,51.9977,45.2507,76.9231,53.0769,75.7692,BAJAJ-AUTO,Bajaj Auto Ltd.,-26.9114,-14.8290,57.5215,19.3794,29.3560,21.1453,72.3505
4,FINANCIAL SERVICES,9,5291,304.1868,6.8785,10.9960,19.7834,0.0342,-64.3630,0.3477,"30,900,715.4917","2,075.9055",51.1527,52.2993,46.6076,100.0000,75.7692,70.7692,HDFC,Housing Development Finance Corporation Ltd.,22.8809,-10.5033,56.0292,17.5758,29.6341,26.6462,66.5325
5,ENERGY,7,5299,268.3712,6.3972,8.8857,18.7663,0.0316,-44.9978,0.3409,"27,782,904.1983","1,079.2297",50.2266,51.9789,44.1010,92.3077,53.4615,64.2308,RELIANCE,Reliance Industries Ltd.,-13.2318,-14.2448,7.6018,17.6679,28.6180,20.0043,21.8466
6,CONSUMER GOODS,6,5303,198.5881,5.3357,7.1393,14.0139,0.0245,-62.5167,0.3807,"11,086,080.6430",520.5570,57.7962,52.5254,44.2141,65.3846,83.8462,63.4615,ASIANPAINT,Asian Paints Ltd.,8.5555,-9.4343,13.4404,11.6038,20.1310,14.4368,22.8747
7,METALS,5,5289,410.0861,8.0728,19.8302,28.1310,0.0465,-71.2039,0.2870,"23,233,202.8245",647.7014,36.9845,51.3758,44.0068,76.9231,20.7692,54.2308,TATASTEEL,Tata Steel Ltd.,-15.9561,-19.5865,102.2296,25.9921,45.0764,26.6053,121.8161
8,FERTILISERS & PESTICIDES,1,4098,159.4789,6.0387,8.5416,32.3134,0.0440,-75.3715,0.1869,"1,692,214.6951",73.6994,55.8502,48.5888,44.1568,11.5385,68.0769,39.6154,UPL,UPL Ltd.,10.1129,-4.8513,76.5635,23.9743,28.4550,30.5316,81.4148
9,CONSTRUCTION,1,3948,95.9316,4.3867,9.1993,26.3066,0.0308,-72.8580,0.1668,"1,917,126.9981",264.4572,47.4611,46.8212,44.0249,34.6154,30.7692,32.5000,LT,Larsen & Toubro Ltd.,-26.8586,-6.0520,12.9594,20.5005,26.3526,23.8566,19.0114


,View Level,Sector / Industry,Date,Year,Quarter,Month,Daily Return,Equal Weighted Index,20D Moving Average,60D Moving Average,Trend Signal,Volume,Turnover (Cr),Average Delivery (%),Average VWAP Gap,Stock Count
0,Market,All Sectors,2000-01-03,2000,1,1,0.0270,102.6999,NaN,NaN,Insufficient History,12398454,439.7286,NaN,2.6106,33
1,Market,All Sectors,2000-01-04,2000,1,1,-0.0014,102.5571,NaN,NaN,Insufficient History,23527330,"1,177.6007",NaN,9.6142,33
2,Market,All Sectors,2000-01-05,2000,1,1,-0.0147,101.0546,NaN,NaN,Insufficient History,47966946,"2,495.4882",NaN,-14.2527,33
3,Market,All Sectors,2000-01-06,2000,1,1,0.0090,101.9629,NaN,NaN,Insufficient History,35021797,"1,625.1511",NaN,-9.0670,33
4,Market,All Sectors,2000-01-07,2000,1,1,-0.0109,100.8514,NaN,NaN,Insufficient History,36313881,"1,165.2534",NaN,-0.6894,33
5,Market,All Sectors,2000-01-10,2000,1,1,-0.0014,100.7087,NaN,NaN,Insufficient History,32614921,"1,614.8508",NaN,8.9491,33
6,Market,All Sectors,2000-01-11,2000,1,1,-0.0158,99.1178,NaN,NaN,Insufficient History,30956990,"1,870.6405",NaN,-11.6976,34
7,Market,All Sectors,2000-01-12,2000,1,1,0.0150,100.6065,NaN,NaN,Insufficient History,23565837,"1,260.6105",NaN,6.9071,34
8,Market,All Sectors,2000-01-13,2000,1,1,-0.0079,99.8070,NaN,NaN,Insufficient History,27401533,"1,279.2984",NaN,-19.1891,34
9,Market,All Sectors,2000-01-14,2000,1,1,-0.0043,99.3772,NaN,NaN,Insufficient History,23201067,"1,145.0384",NaN,12.0103,34


In [5]:
tableau_outputs = {
    'tableau_stock_level.csv': tableau_stock_level,
    'tableau_sector_level.csv': tableau_sector_level,
    'tableau_yearly_trends.csv': tableau_yearly_trends,
    'tableau_risk_segments.csv': tableau_risk_segments,
    'tableau_recommendation_view.csv': tableau_recommendation_view,
}

validation_rows = [
    validate_tableau_dataset(
        name='tableau_stock_level.csv',
        df=tableau_stock_level,
        key_columns=['Stock Symbol'],
        critical_columns=['Annualized Return (%)', 'Volatility (%)', 'Opportunity Score', 'Risk Bucket', 'Segment'],
    ),
    validate_tableau_dataset(
        name='tableau_sector_level.csv',
        df=tableau_sector_level,
        key_columns=['Sector / Industry'],
        critical_columns=['Annualized Return (%)', 'Volatility (%)', 'Opportunity Score', 'COVID Crash Return (%)', 'COVID Recovery Return (%)'],
    ),
    validate_tableau_dataset(
        name='tableau_yearly_trends.csv',
        df=tableau_yearly_trends,
        key_columns=['View Level', 'Sector / Industry', 'Date'],
        critical_columns=['Daily Return', 'Equal Weighted Index', 'Turnover (Cr)', 'Stock Count'],
    ),
    validate_tableau_dataset(
        name='tableau_risk_segments.csv',
        df=tableau_risk_segments,
        key_columns=['Stock Symbol'],
        critical_columns=['Risk Score', 'Risk Bucket', 'Segment', 'Historical VaR 95 (%)', 'Opportunity Score'],
    ),
    validate_tableau_dataset(
        name='tableau_recommendation_view.csv',
        df=tableau_recommendation_view,
        key_columns=['Stock Symbol'],
        critical_columns=['Recommendation Action', 'Opportunity Score', 'Risk Bucket', 'Segment'],
    ),
]
validation_df = pd.DataFrame(validation_rows)

for file_name, dataframe in tableau_outputs.items():
    dataframe.to_csv(DATA_DIR / file_name, index=False)

validation_df.to_csv(OUTPUTS_DIR / 'final_load_validation.csv', index=False)

display(validation_df)

for file_name in tableau_outputs:
    print('saved:', DATA_DIR / file_name)
print('saved:', OUTPUTS_DIR / 'final_load_validation.csv')

,dataset_name,rows,columns,duplicate_key_rows,missing_value_cells,columns_with_missing_values,date_range_start,date_range_end,stock_count,sector_count,kpi_availability_percent,index_like_columns
0,tableau_stock_level.csv,49,27,0,0,No missing columns,2000-01-03,2021-04-30,49,13,100.0000,No index-like columns
1,tableau_sector_level.csv,13,27,0,0,No missing columns,2000-01-03,2021-04-30,0,13,100.0000,No index-like columns
2,tableau_yearly_trends.csv,69627,16,0,7595,"Daily Return, 20D Moving Average, 60D Moving A...",2000-01-03,2021-04-30,0,13,99.5000,No index-like columns
3,tableau_risk_segments.csv,49,26,0,0,No missing columns,2000-01-03,2021-04-30,49,13,100.0000,No index-like columns
4,tableau_recommendation_view.csv,49,20,0,0,No missing columns,2000-01-03,2021-04-30,49,13,100.0000,No index-like columns


saved: /Users/birajitsaikia/Documents/dva-final/DVA_capstone_2/data/processed/tableau_stock_level.csv
saved: /Users/birajitsaikia/Documents/dva-final/DVA_capstone_2/data/processed/tableau_sector_level.csv
saved: /Users/birajitsaikia/Documents/dva-final/DVA_capstone_2/data/processed/tableau_yearly_trends.csv
saved: /Users/birajitsaikia/Documents/dva-final/DVA_capstone_2/data/processed/tableau_risk_segments.csv
saved: /Users/birajitsaikia/Documents/dva-final/DVA_capstone_2/data/processed/tableau_recommendation_view.csv
saved: /Users/birajitsaikia/Documents/dva-final/DVA_capstone_2/outputs/tables/final_load_validation.csv


## Final Notes

- `tableau_stock_level.csv` and `tableau_risk_segments.csv` are stock-grain tables for ranking, scatter plotting, and recommendation filtering.
- `tableau_sector_level.csv` is a sector summary table enriched with COVID crash/recovery comparisons and a best-stock helper field for annotations.
- `tableau_yearly_trends.csv` is a date-grain trend table designed for market and sector filters, yearly trend storytelling, and regime overlays.
- `tableau_recommendation_view.csv` converts the transparent stock segments into dashboard-ready action labels: `Buy`, `Watch`, `Trade Candidate`, and `Avoid`.
- `outputs/tables/final_load_validation.csv` should be reviewed during QA so the final dashboard pack is backed by visible quality checks.

These exports are decision-support inputs for Tableau. They are not meant to replace the underlying analysis notebooks or the written report.